# ACTIVE-TB-X: Official Eval2 Comparison

This compact continuation job reconstructs the two V8 suites missing from the downloaded artifact, then evaluates ACTIVE-TB-X against the exact AutoBench/CorrectBench Eval2 mutant bank.

**Protocol warning:** ACTIVE-TB-X is reference-assisted Track B. AutoBench, CorrectBench, and Direct are spec-only. Their scores are displayed together for test-quality context, but they are not claimed as equal-input generation results.


In [ ]:
# 1) Locate and unpack the attached v9 dataset.
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
matches = list(INPUT.rglob('active_tb_x_sota_eval_v9.zip'))
if len(matches) == 1:
    with zipfile.ZipFile(matches[0]) as zf:
        zf.extractall(WORK)
    ROOT = WORK / 'active_tb_x_sota_eval_v9'
else:
    # Kaggle commonly expands ZIP dataset contents before mounting them.
    markers = list(INPUT.rglob('active_tb_x_sota_eval_v9/official/HDLBits_data.jsonl'))
    assert len(markers) == 1, {'zip_matches': matches, 'markers': markers}
    ROOT = markers[0].parents[1]
PROJECT = ROOT / 'project'
SUITES = ROOT / 'suites'
OFFICIAL = ROOT / 'official'
print('Attached artifact root:', ROOT)
print('Existing V8 suites:', len(list(SUITES.glob('*/b032__active_portfolio_no_llm_suite.json'))))


In [ ]:
# 2) Install only native verification tools and lightweight Python dependencies.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'iverilog', 'verilator', 'yosys'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'vcdvcd'], check=True)
sys.path.insert(0, str(PROJECT))
import ast
ast.parse((PROJECT/'sota_eval'/'eval2_adapter.py').read_text())
print('Toolchain ready')


In [ ]:
# 3) Verify that all V8-supported task suites survived artifact pagination.
supported_suite_ids = {
    path.parent.name for path in SUITES.glob('*/b032__active_portfolio_no_llm_suite.json')
}
documented_unsupported = {'review2015_fancytimer', 'review2015_fsm', 'conwaylife'}
official_ids = {
    json.loads(line)['task_id']
    for line in (OFFICIAL/'HDLBits_data.jsonl').read_text().splitlines() if line.strip()
}
assert official_ids - supported_suite_ids == documented_unsupported, sorted(official_ids - supported_suite_ids)
assert len(supported_suite_ids) == 153, len(supported_suite_ids)
print('V8 supported suites ready:', len(supported_suite_ids))


In [ ]:
# 4) Run official Eval2 with per-task checkpointing.
EVAL_OUT = WORK / 'active_tb_x_official_eval2_v9'
cmd = [
    sys.executable, str(PROJECT/'sota_eval'/'eval2_adapter.py'),
    '--tasks', str(OFFICIAL/'HDLBits_data.jsonl'),
    '--mutants', str(OFFICIAL/'HDLBits_data_mutants.jsonl'),
    '--suites-root', str(SUITES),
    '--output', str(EVAL_OUT),
    '--timeout', '30', '--resume',
]
subprocess.run(cmd, check=True, cwd=PROJECT)


In [ ]:
# 5) Show only the paper-facing comparison and protocol labels.
import json, pandas as pd
active = json.loads((EVAL_OUT/'active_tb_eval2_aggregate.json').read_text())
released = json.loads((ROOT/'baselines'/'released_baseline_runs_aggregate.json').read_text())
rows = []
for method, values in released.items():
    rows.append({
        'method': method,
        'input_access': values['input_access'],
        'tasks': values['tasks_per_run'],
        'runs': values['independent_runs'],
        'Eval2 exact pass@1': values['mean_eval2_exact_pass_at_1'],
        'Eval2 loose pass@1 (>=0.8)': values['mean_eval2_loose_pass_at_1_0_8'],
    })
rows.append({
    'method': active['method'], 'input_access': active['input_access'],
    'tasks': active['tasks_supported'], 'runs': 1,
    'Eval2 exact pass@1': active['eval2_exact_pass_at_1'],
    'Eval2 loose pass@1 (>=0.8)': active['eval2_loose_pass_at_1_0_8'],
})
comparison = pd.DataFrame(rows).sort_values('Eval2 loose pass@1 (>=0.8)', ascending=False)
display(comparison.style.format({'Eval2 exact pass@1':'{:.3f}', 'Eval2 loose pass@1 (>=0.8)':'{:.3f}'}))
comparison.to_csv(EVAL_OUT/'official_eval2_comparison.csv', index=False)
print(json.dumps(active, indent=2))
assert active['tasks_supported'] >= 153, active


In [ ]:
# 6) Save a compact resumable output, excluding disposable tool caches.
archive = WORK / 'active_tb_x_official_eval2_v9_results.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', EVAL_OUT)
print('Result archive:', archive, archive.stat().st_size, 'bytes')
